# Парсинг тендерной документации (PDF/DOCX/DOC) → места / площадь / адрес / тип работ

Задача: по каждой закупке из `data_processed/marker_results_main_clean.csv` скачать/собрать файлы документации и попытаться автоматически вытащить из них:

- places — проектная мощность (мест)
- area_m2 — общая площадь ($м^2$)
- address_text — адрес объекта (текстом)
- work_type_doc — тип работ по документации (construction / caprepair / reconstruction / design_only)

## Как хранятся документы по результатам загрузки

```
docs/
  <registry_number>__<purchase_code>/
    raw/        # как скачалось (pdf/docx/zip/rar)
    extracted/  # распакованное и текст
```

In [2]:
import re
from pathlib import Path

import pandas as pd

ROOT = Path("/Users/arinazajceva/Desktop/диплом")
DATA = ROOT / "data_processed"
DOCS = ROOT / "docs"

RESULTS_MAIN = DATA / "marker_results_main_clean.csv"

print("results_main exists:", RESULTS_MAIN.exists())
print("docs dir exists:", DOCS.exists())


results_main exists: True
docs dir exists: True


In [ ]:
import shutil
import subprocess

RAR_GLOB = "*.rar"

print("7z in PATH:", shutil.which("7z"))
print("7zz in PATH:", shutil.which("7zz"))

raw_rars = sorted((DOCS).rglob(RAR_GLOB))
print("RAR found under docs/:", len(raw_rars))

for p in raw_rars[:5]:
    print("-", p)

7z in PATH: /opt/homebrew/bin/7z
7zz in PATH: None
RAR found under docs/: 1405
- /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/Ä»¿ßá¡¿Ñ «íΩÑ¬Γá ºá¬π»¬¿.rar
- /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/Äìîûè.rar
- /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/Åα«Ñ¬Γ îè.rar
- /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/ïæÉ ( 3 φΓá»).rar
- /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/τáßΓ∞ 6.1 (Åα¿½«ªÑ¡¿Ñ ⁿ1 ¬ «»¿ßá¡¿ε «íΩÑ¬Γá ºá¬π»¬¿).rar


In [ ]:
def try_extract_rar(rar_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = shutil.which("7z") or shutil.which("7zz")
    if not cmd:
        return {"ok": False, "reason": "no_7z_in_path"}

    r = subprocess.run(
        [cmd, "x", f"-o{out_dir}", "-y", str(rar_path)],
        capture_output=True,
        text=True,
    )

    # если это многотомник part1/part2, 7z может ругаться в stderr
    return {
        "ok": r.returncode == 0,
        "returncode": r.returncode,
        "stdout_tail": (r.stdout or "")[-500:],
        "stderr_tail": (r.stderr or "")[-500:],
    }

# Проба распаковки 1-го RAR
if raw_rars:
    test_rar = raw_rars[0]
    test_out = test_rar.parent / "__test_extract"
    res = try_extract_rar(test_rar, test_out)
    print("\nTEST RAR:", test_rar)
    print(res)
    if test_out.exists():
        extracted_files = [p for p in test_out.rglob("*") if p.is_file()]
        print("extracted files:", len(extracted_files))


In [5]:
def try_extract_rar(rar_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    # сначала пробуем 7z, потом 7zz (на некоторых системах есть только он)
    cmd = shutil.which("7z") or shutil.which("7zz")
    if not cmd:
        return {"ok": False, "reason": "no_7z_in_path"}

    r = subprocess.run(
        [cmd, "x", f"-o{out_dir}", "-y", str(rar_path)],
        capture_output=True,
        text=True,
    )

    # если это многотомник part1/part2, 7z может ругаться в stderr
    return {
        "ok": r.returncode == 0,
        "returncode": r.returncode,
        "stdout_tail": (r.stdout or "")[-500:],
        "stderr_tail": (r.stderr or "")[-500:],
    }


test_rar = raw_rars[0]
test_out = test_rar.parent / "__test_extract"
res = try_extract_rar(test_rar, test_out)
print("TEST RAR:", test_rar)
print(res)

TEST RAR: /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/Ä»¿ßá¡¿Ñ «íΩÑ¬Γá ºá¬π»¬¿.rar
{'ok': False, 'returncode': 2, 'stdout_tail': ', 32142 bytes (32 KiB)\n\nExtracting archive: /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/Ä»¿ßá¡¿Ñ «íΩÑ¬Γá ºá¬π»¬¿.rar\n--\nPath = /Users/arinazajceva/Desktop/диплом/docs/101500000322000125__22-30248005212024801001-0031-001-4120-414/extracted/Ä»¿ßá¡¿Ñ «íΩÑ¬Γá ºá¬π»¬¿.rar\nType = Rar\nPhysical Size = 32142\nSolid = -\nBlocks = 3\nMultivolume = -\nVolumes = 1\n\n\nSub items Errors: 2\n\nArchives with Errors: 1\n\nSub items Errors: 2\n', 'stderr_tail': 'ERROR: Unsupported Method : Описание объекта закупки/Описание объекта закупки.doc\nERROR: Unsupported Method : Описание объекта закупки/Приложение №2 к описанию объекта закупки.docx\n'}


In [ ]:
extracted_files = [p for p in test_out.rglob("*") if p.is_file()]
print("extracted files:", len(extracted_files))

extracted files: 2


Массовая распаковка архивов (ZIP -> достать вложенные RAR -> распаковать RAR)

In [7]:
import zipfile
import shutil
import subprocess
from pathlib import Path


def has_cmd(name):
    return shutil.which(name) is not None


def extract_zip_to(zip_path, out_dir):
    try:
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(out_dir)
        return True
    except Exception:
        return False


def extract_rar_to(rar_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    if has_cmd("unar"):
        r = subprocess.run(["unar", "-o", str(out_dir), "-f", str(rar_path)], capture_output=True, text=True)
        return (r.returncode == 0, (r.stderr or r.stdout or "")[ -500 : ])

    if has_cmd("unrar"):
        r = subprocess.run(["unrar", "x", "-o+", "-y", str(rar_path), str(out_dir)], capture_output=True, text=True)
        return (r.returncode == 0, (r.stderr or r.stdout or "")[ -500 : ])

    if has_cmd("7z"):
        r = subprocess.run(["7z", "x", f"-o{out_dir}", "-y", str(rar_path)], capture_output=True, text=True)
        return (r.returncode == 0, (r.stderr or r.stdout or "")[ -500 : ])

    return (False, "no_unar_unrar_7z")


def iter_archives_under_docs():
    exts = {".zip", ".rar"}
    out = []
    for p in DOCS.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            out.append(p)
    return out


def mass_extract_under_docs(limit= None):
    rows = []

    archives = iter_archives_under_docs()
    if limit is not None:
        archives = archives[:limit]

    for a in archives:
        suf = a.suffix.lower()

        if suf == ".zip":
            out_dir = a.parent / "__zip" / a.stem
            ok = extract_zip_to(a, out_dir)
            rows.append({
                "archive": str(a),
                "type": "zip",
                "ok": ok,
                "out_dir": str(out_dir),
                "note": None if ok else "zip_extract_failed",
            })

        elif suf == ".rar":
            out_dir = a.parent / "__rar" / a.stem
            ok, msg = extract_rar_to(a, out_dir)
            note = None if ok else ("rar_extract_failed: " + msg)
            rows.append({
                "archive": str(a),
                "type": "rar",
                "ok": ok,
                "out_dir": str(out_dir),
                "note": note,
            })

    return pd.DataFrame(rows)


print("unrar in PATH:", shutil.which("unrar"))
print("unar in PATH:", shutil.which("unar"))
print("7z in PATH:", shutil.which("7z"))

unrar in PATH: /opt/homebrew/bin/unrar
unar in PATH: None
7z in PATH: /opt/homebrew/bin/7z


In [8]:
log_extract = mass_extract_under_docs(limit=30)
log_extract.head(10)

,archive,type,ok,out_dir,note
0,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,False,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip_extract_failed
1,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
2,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
3,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
4,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
5,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
6,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
7,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None
8,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,False,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip_extract_failed
9,/Users/arinazajceva/Desktop/диплом/docs/108500...,zip,True,/Users/arinazajceva/Desktop/диплом/docs/108500...,None


Функции для чтения документов (PDF, DOCX, DOC, XLSX):

In [ ]:
import fitz  # PyMuPDF
import pdfplumber
from docx import Document
import subprocess

def read_pdf_text(path, max_pages=25):
    "Чтение текста из PDF (сначала PyMuPDF, потом pdfplumber если ничего не нашлось)"
    stop_words = ["площад", "адрес", "мест"]
    text_parts = []

    try:
        doc = fitz.open(str(path))
        pages = range(min(len(doc), max_pages or len(doc)))
        for pg in pages:
            page_text = doc.load_page(pg).get_text("text")
            if page_text and page_text.strip():
                text_parts.append(page_text)
            joined = "\n".join(text_parts).lower()
            if text_parts and any(ch.isdigit() for ch in joined):
                if all(w in joined for w in stop_words):
                    break
        doc.close()
    except Exception:
        pass

    text = "\n\n".join(text_parts).strip()
    if text:
        return text

    # fallback: pdfplumber
    try:
        with pdfplumber.open(str(path)) as pdf:
            for p in pdf.pages[:max_pages or 25]:
                t = p.extract_text() or ""
                if t.strip():
                    text_parts.append(t)
    except Exception:
        pass

    return "\n\n".join(text_parts).strip()


def read_docx_text(path):
    """ Чтение текста из DOCX """
    doc = Document(str(path))
    parts = [p.text for p in doc.paragraphs if p.text and p.text.strip()]
    return "\n".join(parts).strip()


def read_doc_text_macos(path):
    """ Чтение текста из DOC """
    try:
        r = subprocess.run(
            ["/usr/bin/textutil", "-convert", "txt", "-stdout", str(path)],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            return (r.stdout or "").strip()
    except Exception:
        pass
    return ""


def read_xlsx_text(path, max_sheets=5, max_rows=80):
    """ Чтение текста из XLSX """
    try:
        import pandas as pd
        sheets = pd.read_excel(path, sheet_name=None, header=None, nrows=max_rows, engine="openpyxl")
        parts = []
        for i, (name, df) in enumerate(sheets.items()):
            if i >= max_sheets:
                break
            parts.append(str(name))
            for _, row in df.iterrows():
                vals = [str(x) for x in row.tolist() if str(x) not in {"nan", "None"}]
                line = " ".join(v.strip() for v in vals if v and v.strip())
                if line:
                    parts.append(line)
        return "\n".join(parts).strip()
    except Exception:
        return ""


def read_any_text(path):
    suf = path.suffix.lower()
    if suf == ".pdf":
        return read_pdf_text(path)
    if suf == ".docx":
        return read_docx_text(path)
    if suf == ".doc":
        return read_doc_text_macos(path)
    if suf == ".xlsx":
        return read_xlsx_text(path)
    if suf in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    return ""


In [ ]:
_RE_INT = re.compile(r"(\d{2,5})")

def extract_places(text, return_evidence= False):
    text_norm = (text or "").replace("\u00a0", " ")
    t = text_norm.lower()

    def _parse_int_spaced(s):
        try:
            s = (s or "").replace(" ", "")
            if not s.isdigit():
                return None
            return int(s)
        except Exception:
            return None

    num_pat = r"(?:\d{2,5}|\d{1,3}(?:\s\d{3})+)"

    patterns = [
        # явное "школа ... на N мест"
        rf"школ\w*[^\n\r]{{0,120}}?на\s+({num_pat})\s+(?:ученическ\w*\s+)?мест",
        # "на N ученических мест"
        rf"на\s+({num_pat})\s+ученическ\w*\s+мест",
        # "проектная мощность/мощность: N"
        rf"(?:проектн\w*\s+)?мощност\w*[^\n\r]{{0,60}}?({num_pat})\s*(?:мест|чел\w*|учащ\w*|обучающ\w*)?",
        # "количество обучающихся/учащихся: N"
        rf"количеств\w*\s+(?:обучающ\w*|учащ\w*)[^\n\r]{{0,40}}?({num_pat})",
        # "вместимость/наполняемость: N"
        rf"(?:вместим\w*|наполняем\w*)[^\n\r]{{0,60}}?({num_pat})\s*(?:чел\w*|учащ\w*|обучающ\w*|мест)?",
        # "на N учащихся/обучающихся" (без слова "мест")
        rf"на\s+({num_pat})\s+(?:учащ\w*|обучающ\w*)\b",
    ]

    bad_context = [
        "детск",
        "дошкол",
        "групп",
        "койк",
        "посадоч",
        "машино",
        "общежит",
        "спорт",
    ]

    good_context = [
        "школ",
        "ученичес",
        "обучающ",
        "обучен",
        "учащ",
        "общеобраз",
        "сош",
        "лицей",
        "гимназ",
    ]

    candidates = []  # (score, value, snippet)

    def add_candidate(v, ctx, base= 0):
        if v is None:
            return
        if v < 10:
            return

        c = (ctx or "").lower()

        # вместо количества мест можем случайно брать номер пункта договора (14.3) или номер этапа (30)
        is_clause_number = bool(re.search(r"\b\d{1,3}\.\d\b", c))
        is_stage_context = any(w in c for w in ["пункт", "подпункт", "этап", "этапа", "порядков", "номер этап"])
        if v < 100 and (is_clause_number or is_stage_context):
            return

        score = base
        score += 3 if any(g in c for g in good_context) else 0
        score -= 3 if any(b in c for b in bad_context) else 0

        if "ученичес" in c or "обучающ" in c or "учащ" in c:
            score += 1

        if is_clause_number:
            score -= 6
        if is_stage_context:
            score -= 4

        # если число выглядит как год и рядом есть годовой контекст — скорее не мощность
        if re.search(r"\bгод\w*\b|\bг\.\b", c):
            score -= 2

        # если число похоже на год и рядом явно дата/год, то это почти наверняка не места
        # Исключение: если рядом есть явный маркер вместимости ("мест", "учащ") — тогда оставляем
        looks_like_year = 1900 <= v <= 2035
        has_date_context = bool(re.search(r"\b20\d\d\s*(?:г\.|год)\b|\b\d{1,2}[\./]\d{1,2}[\./]20\d\d\b", c))
        has_capacity_words = ("мест" in c) or ("учащ" in c) or ("обуча" in c)
        if looks_like_year and has_date_context and (not has_capacity_words):
            return

        # шаблон даты "202_ г." (ловится как 202)
        if v in {200, 201, 202, 203, 204, 205, 206, 207, 208, 209} and re.search(r"\b20\d_\s*г\b|\b20\d_\s*г\.", c):
            return

        snippet = re.sub(r"\s+", " ", (ctx or "")).strip()
        candidates.append((score, v, snippet))

    # 1) regex-слой по всему тексту
    for pat in patterns:
        for m in re.finditer(pat, t):
            v = _parse_int_spaced(m.group(1))
            if v is None:
                continue

            ctx = text_norm[max(0, m.start() - 120) : min(len(text_norm), m.end() + 120)]
            add_candidate(v, ctx)

    # 2) строчный/табличный слой: мощность бывает в ТЭП-таблицах
    num_re = re.compile(r"(\d{1,3}(?:\s\d{3})+|\d{2,5})(?!\.\d)")
    lines_raw = (text or "").replace("\u00a0", " ").splitlines()
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in lines_raw if ln and ln.strip()]

    def looks_like_places_line(s):
        s = (s or "").lower()
        return (
            ("мощност" in s)
            or ("ученичес" in s)
            or ("обучающ" in s)
            or ("учащ" in s)
            or ("вместим" in s)
            or ("наполняем" in s)
            or ("мест" in s and "машино" not in s)
        )

    for i, ln in enumerate(lines):
        if not looks_like_places_line(ln):
            continue

        next_ln = lines[i + 1] if i + 1 < len(lines) else ""
        window_raw = f"{ln} {next_ln}"
        window = window_raw.lower()

        anchor_pos = -1
        for kw in ["мощност", "ученичес", "обучающ", "учащ", "вместим", "наполняем", "мест"]:
            p = window.find(kw)
            if p >= 0:
                anchor_pos = p
                break

        window_after = window[anchor_pos:] if anchor_pos >= 0 else window
        m = num_re.search(window_after)
        if not m:
            continue

        v = _parse_int_spaced(m.group(1))
        if v is None:
            continue

        add_candidate(v, window_raw, base=1)

    if not candidates:
        return (None, None) if return_evidence else None

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    best = candidates[0]
    return (best[1], best[2]) if return_evidence else best[1]

Извлечение мест из названия публикации:

In [ ]:
def extract_places_from_title(title, return_evidence= False):
    import re

    text = (title or "").replace("\u00a0", " ").strip()
    if not text:
        return (None, None) if return_evidence else None

    t_full = text.lower()

    school_markers = ["школ", "сош", "лицей", "гимназ", "общеобраз"]
    if not any(m in t_full for m in school_markers):
        return (None, None) if return_evidence else None

    preschool_markers = ["детск", "дошкол", "сад", "доу", "групп"]
    dorm_markers = ["интернат", "общежит"]

    num_pat = r"(?:\d{2,5}|\d{1,3}(?:\s\d{3})+)"

    def _parse_int_spaced(s):
        s = (s or "").replace(" ", "")
        return int(s) if s.isdigit() else None

    def _split_title(s):
        parts = re.split(r"[;,]|\(|\)|\[|\]|\{|\}|\n|\r|\t|\s[—–-]\s|:\s", s)
        parts = [p.strip() for p in parts if p and p.strip()]
        return parts if parts else [s]

    def _reject_year_like(v, ctx_low):
        looks_like_year = 2000 <= v <= 2035
        has_date_context = bool(
            re.search(r"\b20\d\d\s*(?:г\.|год)\b|\b\d{1,2}[\./]\d{1,2}[\./]20\d\d\b", ctx_low)
        )
        return bool(looks_like_year and has_date_context)

    candidates = []  # (score, value, snippet)

    def add(v, snippet):
        if v is None or v < 10:
            return

        sn = (snippet or "").strip()
        c = sn.lower()

        if _reject_year_like(v, c):
            return

        score = 0
        score += 6 if any(m in c for m in school_markers) else 0
        score += 3 if ("ученичес" in c or "учащ" in c or "обуча" in c) else 0

        if any(m in c for m in preschool_markers):
            score -= 8
        if any(m in c for m in dorm_markers):
            score -= 6

        sn = re.sub(r"\s+", " ", sn).strip()
        candidates.append((score, v, sn))

    # сначала пробуем паттерн по всему заголовку
    pat_school = re.compile(
        rf"(школ\w*|общеобраз\w*|сош|лицей|гимназ\w*)[^\n\r]{{0,200}}?на\s+({num_pat})\s+(?:ученическ\w*\s+)?мест"
    )
    for m in pat_school.finditer(t_full):
        v = _parse_int_spaced(m.group(2))
        if v is not None:
            add(v, text)

    # дальше работаем по кускам
    for part in _split_title(text):
        p_low = part.lower()
        if not any(m in p_low for m in school_markers):
            continue

        m = re.search(r"на\s+(\d{2,4})\s*/\s*(\d{2,4})\s+мест", p_low)
        if m:
            v1 = _parse_int_spaced(m.group(1))
            v2 = _parse_int_spaced(m.group(2))
            if v1 is not None:
                add(v1, part)
            if v2 is not None:
                add(v2, part)

        # на N мест
        for m in re.finditer(rf"на\s+({num_pat})\s+(?:ученическ\w*\s+)?мест", p_low):
            v = _parse_int_spaced(m.group(1))
            if v is not None:
                add(v, part)

        # на N учащихся/обучающихся
        for m in re.finditer(rf"на\s+({num_pat})\s+(?:учащ\w*|обучающ\w*)\b", p_low):
            v = _parse_int_spaced(m.group(1))
            if v is not None:
                add(v, part)

    if not candidates:
        return (None, None) if return_evidence else None

    # выбираем по score, при равном — по величине (школа обычно больше детсада)
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    best = candidates[0]
    return (best[1], best[2]) if return_evidence else best[1]


def _parse_num_ru(s):
    s = (s or "").replace("\u00a0", " ").strip()
    s = s.replace(" ", "").replace(",", ".")
    try:
        return float(s)
    except Exception:
        return None

Извлечение площади:

In [ ]:

def extract_area_m2(text, return_evidence= False):
    text_norm = (text or "").replace("\u00a0", " ")
    t = text_norm.lower()

    exclude = [
        "участка",
        "земельного участка",
        "земельный участок",
        "участок",
        "застройки",
        "охранная зона",
        "сервитут",
        "газопровод",
    ]

    roomish = [
        "кабинет",
        "помещен",
        "коридор",
        "вестибюл",
        "спортзал",
        "актов",
        "столов",
        "сануз",
        "лестниц",
        "этаж",
        "блок",
    ]

    unit_tokens = ["м2", "м 2", "м²", "кв.м", "кв. м", "кв м", "кв.мет", "кв.метр"]

    def has_units(s):
        s = (s or "").lower()
        return any(u in s for u in unit_tokens)

    def score_candidate(ctx, v, units):
        c = (ctx or "").lower()
        score = 0

        if any(w in c for w in exclude):
            score -= 10
        if any(w in c for w in roomish):
            score -= 4

        if "общ" in c and "площад" in c:
            score += 6
        if "здан" in c and "площад" in c:
            score += 4
        if "объект" in c and "площад" in c:
            score += 3
        if "школ" in c and "площад" in c:
            score += 2
        if units:
            score += 1

        # рядом что-то про деньги - не подходит
        if (not units) and any(w in c for w in ["руб", "₽", "цена", "стоим", "контракт", "нмцк", "аванс", "оплат", "сумм"]):
            score -= 8

        # частый мусор: методички вида "значения площади – в м2 ... 0,00" (это не площадь объекта)
        if ("значен" in c and "площад" in c) and (
            "округлен" in c or "знак" in c or "формат" in c or "единиц" in c
        ):
            score -= 12

        # слишком маленькие значения (100, 150) чаще относятся к помещениям,
        # поэтому без "общая площадь/здания/объекта" штраф
        if v < 300 and not (
            ("общ" in c and "площад" in c)
            or ("здан" in c and "площад" in c)
            or ("объект" in c and "площад" in c)
        ):
            score -= 6

        # числа, похожие на год, вряд ли площадь
        if 1900 <= v <= 2035:
            if re.search(r"\b20\d\d\s*(?:г\.|год)\b|\b\d{1,2}[\./]\d{1,2}[\./]20\d\d\b", c):
                score -= 8

        if v >= 1000:
            score += 1

        return score

    candidates = []  # (score, value, snippet)

    # 1) regex-слой
    patterns = [
        # прямой порядок: "общая площадь ... м2"
        r"общ\w*\s+площад\w*[^0-9]{0,120}([0-9\s]{3,9}(?:[\.,][0-9]{1,2})?)\s*(?:м2|м\s*2|м²|кв\.?\s*м)",
        r"площад\w*\s+здани\w*[^0-9]{0,120}([0-9\s]{3,9}(?:[\.,][0-9]{1,2})?)\s*(?:м2|м\s*2|м²|кв\.?\s*м)",
        r"площад\w*\s+объект\w*[^0-9]{0,120}([0-9\s]{3,9}(?:[\.,][0-9]{1,2})?)\s*(?:м2|м\s*2|м²|кв\.?\s*м)",
        # обратный порядок: "... м2 общая площадь"
        r"([0-9\s]{3,9}(?:[\.,][0-9]{1,2})?)\s*(?:м2|м\s*2|м²|кв\.?\s*м)[^\n\r]{0,80}общ\w*\s+площад\w*",
        r"([0-9\s]{3,9}(?:[\.,][0-9]{1,2})?)\s*(?:м2|м\s*2|м²|кв\.?\s*м)[^\n\r]{0,80}площад\w*\s+здани\w*",
        # "Sобщ"
        r"s\s*общ\w*\s*[:=\-]?\s*([0-9\s]{3,9}(?:[\.,][0-9]{1,2})?)\s*(?:м2|м\s*2|м²|кв\.?\s*м)",
    ]

    for pat in patterns:
        for m in re.finditer(pat, t):
            v = _parse_num_ru(m.group(1))
            if v is None:
                continue
            if v <= 0:
                continue
            ctx = text_norm[max(0, m.start() - 120) : min(len(text_norm), m.end() + 120)]
            snippet = re.sub(r"\s+", " ", ctx).strip()
            candidates.append((score_candidate(ctx.lower(), v, units=True), v, snippet))

    # 2) строковый слой (число может быть на следующей строке)
    num_re = re.compile(r"(\d{3,}(?:\s\d{3})*(?:[\.,]\d{1,2})?)")

    lines_raw = (text or "").replace("\u00a0", " ").splitlines()
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in lines_raw if ln and ln.strip()]

    for i, ln in enumerate(lines):
        low = ln.lower()
        if "площад" not in low:
            continue
        if "площадк" in low:
            continue
        if any(w in low for w in exclude):
            continue

        next_ln = lines[i + 1] if i + 1 < len(lines) else ""
        window = f"{ln} {next_ln}".lower()

        pos = low.find("площад")
        window_after = f"{ln} {next_ln}" if next_ln else ln
        window_after = window_after[pos:] if pos >= 0 else window_after

        m = num_re.search(window_after)
        if not m:
            # число стоит не сразу после слова "площадь", пробуем взять первое "похоже-на-площадь" число из окна
            if has_units(window):
                m2 = num_re.search(window)
                if m2:
                    v2 = _parse_num_ru(m2.group(1))
                    if v2 is not None and v2 > 0:
                        snippet2 = re.sub(r"\s+", " ", window).strip()
                        candidates.append((score_candidate(window, v2, units=True) - 1, v2, snippet2))
            continue

        v = _parse_num_ru(m.group(1))
        if v is None:
            continue

        units_here = has_units(window)

        snippet = re.sub(r"\s+", " ", window).strip()
        candidates.append((score_candidate(window, v, units=units_here), v, snippet))

    # 3) заголовок + число на следующей строке (в таблице)
    header_markers = [
        "общая площадь",
        "площадь здания",
        "площадь объекта",
        "sобщ",
        "s общ",
    ]

    for i, ln in enumerate(lines):
        low = ln.lower()
        if "площадк" in low:
            continue
        if not any(h in low for h in header_markers):
            continue
        if not has_units(low):
            continue

        # смотрим 1-2 следующие строки: там обычно стоит число
        for j in (i + 1, i + 2):
            if j >= len(lines):
                continue
            nxt = lines[j]
            m = num_re.search(nxt)
            if not m:
                continue

            v = _parse_num_ru(m.group(1))
            if v is None or v <= 0:
                continue

            ctx = (ln + " " + nxt).lower()
            snippet = re.sub(r"\s+", " ", (ln + " " + nxt)).strip()
            candidates.append((score_candidate(ctx, v, units=True) + 1, v, snippet))
            break

    if not candidates:
        return (None, None) if return_evidence else None

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    best = candidates[0]
    return (best[1], best[2]) if return_evidence else best[1]

Извлечение адреса:

In [ ]:
def extract_address_text(text):
    t = (text or "").replace("\u00a0", " ")

    patterns = [
        r"по\s+адресу\s*[:\-]\s*(.{10,260})",
        r"адрес\s+(?:объект\w*|строительств\w*|выполнени\w*\s+работ)\s*[:\-]\s*(.{10,260})",
        r"место\s*нахождени\w*\s*(?:объект\w*)?\s*[:\-]\s*(.{10,260})",
        r"местонахождени\w*\s*(?:объект\w*)?\s*[:\-]\s*(.{10,260})",
        r"место\s+выполнени\w*\s+работ\s*[:\-]\s*(.{10,260})",
        r"место\s+оказани\w*\s+услуг\s*[:\-]\s*(.{10,260})",
        r"место\s+поставк\w*\s*[:\-]\s*(.{10,260})",
        r"располож\w*\s+по\s+адресу\s*(.{10,260})",
    ]

    def clean_addr(s):
        s = s.strip(" \t\"“”„»«")
        s = re.split(r"\n|\r|\)\s*$|;\s*", s)[0].strip(" \t\"“”„»«")
        return s

    for pat in patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if not m:
            continue
        addr = clean_addr(m.group(1))
        if not addr:
            continue
        low = addr.lower()
        if low.startswith("http") or "http://" in low or "https://" in low:
            continue
        if "@" in low:
            continue
        return addr

    # fallback
    lines_raw = (text or "").replace("\u00a0", " ").splitlines()
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in lines_raw if ln and ln.strip()]

    keys = [
        "адрес",
        "по адресу",
        "место выполнения работ",
        "место оказания услуг",
        "место поставки",
        "место нахождения",
        "местонахождение",
    ]

    for i, ln in enumerate(lines):
        low = ln.lower()
        if not any(k in low for k in keys):
            continue

        candidate = ""
        if ":" in ln:
            candidate = ln.split(":", 1)[1].strip()
        elif ln.endswith(":" ) and i + 1 < len(lines):
            candidate = lines[i + 1].strip()

        candidate = clean_addr(candidate)
        if not candidate:
            continue

        low2 = candidate.lower()
        if low2.startswith("http") or "http://" in low2 or "https://" in low2:
            continue

        return candidate

    return None

In [ ]:
def confidence_from_fields(places, area_m2, address_text):
    score = 0
    score += 1 if places is not None else 0
    score += 1 if area_m2 is not None else 0
    score += 1 if address_text else 0
    if score >= 3:
        return "high"
    if score == 2:
        return "medium"
    return "low"

In [ ]:
def normalize_pc(v):
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass

    s = str(v).strip()
    return "" if s.lower() == "nan" else s


def tender_root_for_row(registry_number, purchase_code):
    p = DOCS / f"{registry_number}__{purchase_code}"
    if p.exists():
        return p

    if not purchase_code:
        p2 = DOCS / f"{registry_number}__nan"
        if p2.exists():
            return p2

    return p

In [ ]:
KEYWORDS_DOC_PRIORITY = [
    "заключение экспертизы",
    "государственной экспертизы",
    "экспертиз",
    "технико-экономическ",
    "сведения о технико",
    "тэп",
    "поясн",
    "пз",
    "описание объекта",
    "описание объекта закупки",
    "техническое задание",
    "тз",
]

KEYWORDS_DOC_DEPRIORITY = []


def iter_doc_files(tender_root, raw_only_success= True):
    """ Итерирует по всем документам в корне закупки """
    out = []

    ext_dir = tender_root / "extracted"
    raw_dir = tender_root / "raw"

    exts = {".pdf", ".docx", ".doc", ".txt", ".xlsx"}

    # 1) extracted — всегда
    if ext_dir.exists():
        for p in sorted(ext_dir.rglob("*")):
            if p.is_file() and p.suffix.lower() in exts:
                out.append(p)

    # 2) raw — опционально только успешные
    if raw_dir.exists():
        allowed_raw_suffixes = None

        if raw_only_success:
            manifest = raw_dir / "__manifest.csv"
            if manifest.exists():
                allowed = set()
                try:
                    import csv

                    with manifest.open("r", encoding="utf-8", errors="replace", newline="") as f:
                        r = csv.DictReader(f)
                        for row in r:
                            st = (row.get("status") or "").strip().lower()
                            fn = (row.get("filename") or "").strip()
                            dl = (row.get("downloaded") or "").strip().lower()
                            if not fn:
                                continue
                            if dl not in {"true", "1", "yes"}:
                                continue
                            if st == "успешно":
                                # файлы в raw сохраняются как "NN__<name>"
                                allowed.add(f"__{fn}")
                    allowed_raw_suffixes = allowed
                except Exception:
                    allowed_raw_suffixes = None

        for p in sorted(raw_dir.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix.lower() not in exts:
                continue

            if allowed_raw_suffixes is not None:
                if not any(p.name.endswith(suf) for suf in allowed_raw_suffixes):
                    continue

            out.append(p)

    return out

In [ ]:
def is_priority_doc(path):
    name = path.name.lower()
    return any(k in name for k in KEYWORDS_DOC_PRIORITY)


def file_size_bytes(path):
    try:
        return path.stat().st_size
    except Exception:
        return 0


def sort_doc_files(files: list[Path]):
    return sorted(files, key=lambda p: (is_priority_doc(p), file_size_bytes(p)), reverse=True)

Парсинг закупки:

In [ ]:
def parse_one_tender(row, max_files= 30):
    reg = str(row["registry_number"])
    pc = normalize_pc(row.get("purchase_code"))
    tender_root = tender_root_for_row(reg, pc)

    all_files = sort_doc_files(iter_doc_files(tender_root, raw_only_success=False))
    files = all_files[:max_files]

    def has_teps_markers(text):
        t = (text or "").lower()
        if "технико-эконом" in t or "тэп" in t:
            return True
        if "мощност" in t:
            return True
        if "общая площад" in t:
            return True
        if "площад" in t and ("м2" in t or "м 2" in t or "м²" in t or "кв" in t):
            return True
        return False

    places = None
    area_m2 = None
    address_text = None

    places_source_file = None
    places_snippet = None
    area_m2_source_file = None
    area_m2_snippet = None

    # сначала пытаемся достать места из названия публикации
    title = str(row.get("publication_name") or "")
    v_title, snip_title = extract_places_from_title(title, return_evidence=True)
    if v_title is not None:
        places = v_title
        places_source_file = "__title__"
        places_snippet = snip_title

    used_files = []
    bad_files = []

    def read_one(f):
        try:
            return read_any_text(f)
        except Exception as e:
            bad_files.append(f"{f.name} :: {type(e).__name__}")
            return ""

    # проход по верхним файлам
    for f in files:
        used_files.append(f.name)
        t = read_one(f)

        if not t:
            try:
                if f.stat().st_size == 0:
                    bad_files.append(f"{f.name} :: empty_file")
                else:
                    bad_files.append(f"{f.name} :: empty_text")
            except Exception:
                bad_files.append(f"{f.name} :: empty_text")
            continue

        if places is None:
            v, snip = extract_places(t, return_evidence=True)
            if v is not None:
                places = v
                places_source_file = f.name
                places_snippet = snip

        if area_m2 is None:
            v, snip = extract_area_m2(t, return_evidence=True)
            if v is not None:
                area_m2 = v
                area_m2_source_file = f.name
                area_m2_snippet = snip

        if address_text is None:
            address_text = extract_address_text(t)

        if places is not None and area_m2 is not None:
            break

    def has_area_markers(text):
        tt = (text or "").lower()
        if "площад" not in tt:
            return False
        if "м²" in tt or "кв" in tt or "кв.м" in tt or "м2" in tt or "м 2" in tt:
            return True
        if "sобщ" in tt or "s общ" in tt:
            return True
        return False

    # добор по остальным файлам
    if places is None or area_m2 is None:
        for f in all_files[max_files:]:
            if f.name in used_files:
                continue

            t = read_one(f)
            if not t:
                continue

            if places is None and has_teps_markers(t):
                used_files.append(f.name)
                v, snip = extract_places(t, return_evidence=True)
                if v is not None:
                    places = v
                    places_source_file = f.name
                    places_snippet = snip

            if area_m2 is None and has_area_markers(t):
                if f.name not in used_files:
                    used_files.append(f.name)
                v, snip = extract_area_m2(t, return_evidence=True)
                if v is not None:
                    area_m2 = v
                    area_m2_source_file = f.name
                    area_m2_snippet = snip

            if address_text is None:
                address_text = extract_address_text(t)

            if places is not None and area_m2 is not None:
                break

    if address_text is None:
        address_text = extract_address_text(str(row.get("publication_name", "")))

    conf = confidence_from_fields(places, area_m2, address_text)

    na_reason = None
    if not all_files:
        na_reason = "no_docs_found"
    elif places is None and area_m2 is None and address_text is None:
        na_reason = "docs_unreadable_or_no_text"

    return {
        "registry_number": reg,
        "purchase_code": pc,
        "publication_url": row.get("publication_url"),
        "publication_name": row.get("publication_name"),
        "region": row.get("Регион поставки"),
        "city": row.get("Город поставки"),
        "date_published": row.get("Дата публикации"),
        "docs_root": str(tender_root),
        "places": places,
        "places_source_file": places_source_file,
        "places_snippet": places_snippet,
        "area_m2": area_m2,
        "area_m2_source_file": area_m2_source_file,
        "area_m2_snippet": area_m2_snippet,
        "address_text": address_text,
        "confidence": conf,
        "na_reason": na_reason,
        "used_files": "; ".join(used_files[:10]) if used_files else None,
        "bad_files": "; ".join(bad_files[:10]) if bad_files else None,
        "bad_files_n": len(bad_files),
    }

Загружаю список закупок:

In [ ]:
results_main = pd.read_csv(RESULTS_MAIN)

print("rows:", len(results_main))
cols = [c for c in ["registry_number", "purchase_code", "publication_name", "publication_url"] if c in results_main.columns]
results_main[cols].head()

rows: 157


,registry_number,purchase_code,publication_name,publication_url
0,6448182,NaN,Капитальное строительство центра образования е...,https://analytics.marker-zakupki.ru/Card/Lot/1...
1,8815987,NaN,Выполнение работ по оценке технического состоя...,https://analytics.marker-zakupki.ru/Card/Lot/1...
2,101500000322000125,22-30248005212024801001-0031-001-4120-414,Школа на 550 мест с интернатом на 140 мест в с...,https://analytics.marker-zakupki.ru/Card/Lot/1...
3,101500000322000183,22-20278176470027601001-0348-001-4120-414,"Выполнение строительно-монтажных, пусконаладоч...",https://analytics.marker-zakupki.ru/Card/Lot/1...
4,101500000322000257,22-20278176470027601001-0348-002-4120-414,"Выполнение строительно-монтажных, пусконаладоч...",https://analytics.marker-zakupki.ru/Card/Lot/1...


Пробный прогон на одной закупке — проверяю, что парсинг работает:

In [ ]:
N = 45
sample = results_main.head(1)

rows = []
for _, r in sample.iterrows():
    rows.append(parse_one_tender(r))

parsed_preview = pd.DataFrame(rows)
parsed_preview

,registry_number,purchase_code,publication_url,publication_name,region,city,date_published,docs_root,places,places_source_file,places_snippet,area_m2,area_m2_source_file,area_m2_snippet,address_text,confidence,na_reason,used_files,bad_files,bad_files_n
0,6448182,,https://analytics.marker-zakupki.ru/Card/Lot/1...,Капитальное строительство центра образования е...,Оренбургская область,Северный район,45146.256655,/Users/arinazajceva/Desktop/диплом/docs/6448182__,None,None,None,None,None,None,"Оренбургская область, Северный район, с. Север...",low,no_docs_found,None,None,0


Полный прогон по всем закупкам:

In [ ]:
OUT = DATA / "marker_docs_parsed_auto.csv"

rows = []
for _, r in results_main.iterrows():
    rows.append(parse_one_tender(r))

parsed_all = pd.DataFrame(rows)
parsed_all.to_csv(OUT, index=False)

print('saved:', OUT)


/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: '01-01-01 Вынос электросети КЛ-1'!A:N.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: '01-02-01 Демонтаж существующих '!A:N.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: '02-01 Строительство школы , (в1'!A:H.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: '02-01 Строительство школы , (вр'!A:H.
  warn(f"Pr

saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_parsed_auto.csv


Проверяю качество извлечения — смотрю на подозрительные значения

In [ ]:
def _ratio(area_m2, places):
    if area_m2 is None or places is None:
        return None
    try:
        a, p = float(area_m2), float(places)
        return a / p if p > 0 else None
    except Exception:
        return None


audit = parsed_all.copy()


def _is_depriority_filename(name):
    s = (name or "").lower()
    if not s:
        return False
    return any(k in s for k in KEYWORDS_DOC_DEPRIORITY)


audit["places_from_depriority"] = audit["places_source_file"].apply(_is_depriority_filename)
audit["area_m2_from_depriority"] = audit["area_m2_source_file"].apply(_is_depriority_filename)


audit["area_per_place"] = [
    _ratio(a, p) for a, p in zip(audit.get("area_m2"), audit.get("places"))
]

audit["flag_places_out_of_range"] = audit["places"].apply(
    lambda x: (pd.notna(x) and (float(x) < 50 or float(x) > 3000))
)
audit["flag_area_out_of_range"] = audit["area_m2"].apply(
    lambda x: (pd.notna(x) and (float(x) < 500 or float(x) > 200000))
)

In [ ]:
# отношение площадь/место - перцентили
P_LOW = 0.05
P_HIGH = 0.95

ratio_series = pd.to_numeric(audit["area_per_place"], errors="coerce").dropna()
ratio_low = float(ratio_series.quantile(P_LOW)) if len(ratio_series) else None
ratio_high = float(ratio_series.quantile(P_HIGH)) if len(ratio_series) else None

audit["ratio_p_low"] = ratio_low
audit["ratio_p_high"] = ratio_high

def _ratio_out_of_percentile_band(r):
    try:
        if r is None or pd.isna(r):
            return False
        if ratio_low is None or ratio_high is None:
            return False
        rr = float(r)
        return rr < ratio_low or rr > ratio_high
    except Exception:
        return False

In [ ]:
audit["flag_ratio_out_of_range"] = audit["area_per_place"].apply(_ratio_out_of_percentile_band)

# оценка риска: чем больше флагов, тем выше приоритет на просмотр
flag_cols = ["flag_places_out_of_range", "flag_area_out_of_range", "flag_ratio_out_of_range"]
audit["audit_flags_n"] = audit[flag_cols].sum(axis=1)

summary = {
    "rows": int(len(audit)),
    "places_filled": int(audit["places"].notna().sum()),
    "area_m2_filled": int(audit["area_m2"].notna().sum()),
    "both_filled": int((audit["places"].notna() & audit["area_m2"].notna()).sum()),
    "ratio_flagged": int(audit["flag_ratio_out_of_range"].sum()),
    "any_flagged": int((audit["audit_flags_n"] > 0).sum()),
    "places_from_depriority": int(audit["places_from_depriority"].sum()),
    "area_m2_from_depriority": int(audit["area_m2_from_depriority"].sum()),
    "ratio_p_low": ratio_low,
    "ratio_p_high": ratio_high,
    "ratio_p_low_q": P_LOW,
    "ratio_p_high_q": P_HIGH,
}
summary["places_coverage"] = summary["places_filled"] / summary["rows"] if summary["rows"] else 0
summary["area_m2_coverage"] = summary["area_m2_filled"] / summary["rows"] if summary["rows"] else 0
summary["both_coverage"] = summary["both_filled"] / summary["rows"] if summary["rows"] else 0

summary_df = pd.DataFrame([summary])
AUDIT_SUMMARY = DATA / "marker_docs_parse_audit_summary.csv"
AUDIT_FLAGS = DATA / "marker_docs_parse_audit_flags.csv"

TOP_SUSPICIOUS = DATA / "marker_docs_parse_audit_top30_suspicious.csv"
TOP_CONFIDENT = DATA / "marker_docs_parse_audit_top30_confident.csv"

In [ ]:
susp = audit.copy()
if ratio_low is not None and ratio_high is not None:
    mid = (ratio_low + ratio_high) / 2
    susp["abs_ratio_deviation"] = susp["area_per_place"].apply(
        lambda r: (abs(float(r) - mid) if (r is not None and pd.notna(r)) else None)
    )
else:
    susp["abs_ratio_deviation"] = None

susp_top = (
    susp.sort_values(by=["audit_flags_n", "abs_ratio_deviation"], ascending=[False, False])
    .head(30)
)

conf_top = (
    audit[(audit["audit_flags_n"] == 0) & audit["places"].notna() & audit["area_m2"].notna()]
    .copy()
    .head(30)
)

audit.to_csv(AUDIT_FLAGS, index=False)
summary_df.to_csv(AUDIT_SUMMARY, index=False)
susp_top.to_csv(TOP_SUSPICIOUS, index=False)
conf_top.to_csv(TOP_CONFIDENT, index=False)

# строки, где площадь не найдена, но документы читались
MISSING_TOP30 = DATA / "marker_docs_area_missing_top30.csv"
MISSING_ALL = DATA / "marker_docs_area_missing_all.csv"

miss_all = audit[(audit["area_m2"].isna()) & audit["used_files"].notna()].copy()
miss_all.to_csv(MISSING_ALL, index=False)

miss_all.head(30).to_csv(MISSING_TOP30, index=False)

In [ ]:
print("audit saved:", AUDIT_SUMMARY)
print("flags saved:", AUDIT_FLAGS)
print("top suspicious saved:", TOP_SUSPICIOUS)
print("top confident saved:", TOP_CONFIDENT)
print("missing-area top30 saved:", MISSING_TOP30)
print("missing-area all saved:", MISSING_ALL)
summary_df

audit saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_parse_audit_summary.csv
flags saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_parse_audit_flags.csv
top suspicious saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_parse_audit_top30_suspicious.csv
top confident saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_parse_audit_top30_confident.csv
missing-area top30 saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_area_missing_top30.csv
missing-area all saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_area_missing_all.csv


,rows,places_filled,area_m2_filled,both_filled,ratio_flagged,any_flagged,places_from_depriority,area_m2_from_depriority,ratio_p_low,ratio_p_high,ratio_p_low_q,ratio_p_high_q,places_coverage,area_m2_coverage,both_coverage
0,157,155,129,129,13,67,0,0,0.004431,144.142857,0.05,0.95,0.987261,0.821656,0.821656


- `places_coverage`: доля строк, где удалось извлечь `places`
- `area_m2_coverage`: доля строк, где удалось извлечь `area_m2`
- `ratio_flagged`: сколько строк помечено как "подозрительные" по отношению `area_m2/places` по перцентилям (границы записаны в `marker_docs_parse_audit_summary.csv` как `ratio_p_low`, `ratio_p_high` и квантили `ratio_p_low_q`, `ratio_p_high_q`)


## Автооценка качества `places`

Идея: в одной закупке число мест обычно повторяется в нескольких документах (ПЗ, экспертиза, ТЗ, описание объекта).
Если по разным файлам устойчиво всплывает одно и то же значение — это сильный возможный корректности извлечения.

План:
- перечитать ограниченное число файлов из `docs_root/extracted/`
- извлечь `places` из каждого файла
- сколько источников подтверждают одно и то же значение и есть ли конфликты
- summary + топ конфликтов сохраняются


In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data_processed"
INP = DATA / "marker_docs_parsed_auto_targeted_ocr.csv"

if not INP.exists():
    raise FileNotFoundError(f"Не найден файл: {INP}")

need = ["extract_places", "iter_doc_files", "sort_doc_files", "read_any_text"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(
        "Не найдены функции из предыдущих ячеек: " + ", ".join(missing) + ".\n"
        "Запусти парсинг-ячейки выше (где определяются read_any_text / iter_doc_files / extract_places)."
    )


df = pd.read_csv(INP)

In [ ]:
def _audit_places_one(docs_root, max_files= 18):
    if not docs_root or str(docs_root).strip() == "":
        return {
            "places_mentions_total": 0,
            "places_distinct_n": 0,
            "places_mode": None,
            "places_mode_count": 0,
            "places_mode_share": None,
            "places_values": "",
            "places_mode_sources": "",
        }

    tender_root = Path(str(docs_root))
    if not tender_root.exists():
        return {
            "places_mentions_total": 0,
            "places_distinct_n": 0,
            "places_mode": None,
            "places_mode_count": 0,
            "places_mode_share": None,
            "places_values": "",
            "places_mode_sources": "",
        }

    files = sort_doc_files(iter_doc_files(tender_root, raw_only_success=False))
    files = [f for f in files if f.suffix.lower() in {".pdf", ".docx", ".doc", ".xlsx", ".txt", ".md"}]
    files = files[:max_files]

    mentions = []  # (value, filename)

    for f in files:
        try:
            txt = read_any_text(f)
        except Exception:
            continue

        if not txt or len(txt.strip()) < 40:
            continue

        v, _snip = extract_places(txt, return_evidence=True)
        if v is None:
            continue

        try:
            v_int = int(v)
        except Exception:
            continue

        mentions.append((v_int, f.name))

    if not mentions:
        return {
            "places_mentions_total": 0,
            "places_distinct_n": 0,
            "places_mode": None,
            "places_mode_count": 0,
            "places_mode_share": None,
            "places_values": "",
            "places_mode_sources": "",
        }

    cnt = Counter([v for v, _ in mentions])
    mode_v, mode_n = cnt.most_common(1)[0]
    total = sum(cnt.values())
    share = (mode_n / total) if total else None

    mode_sources = sorted({fn for v, fn in mentions if v == mode_v})
    values_str = "; ".join([f"{v}:{n}" for v, n in sorted(cnt.items(), key=lambda x: (-x[1], x[0]))])

    return {
        "places_mentions_total": int(total),
        "places_distinct_n": int(len(cnt)),
        "places_mode": int(mode_v),
        "places_mode_count": int(mode_n),
        "places_mode_share": float(share) if share is not None else None,
        "places_values": values_str,
        "places_mode_sources": "; ".join(mode_sources),
    }

Прогон аудита:

In [ ]:
OUT_SUM = DATA / "marker_places_audit_summary.csv"
OUT_ROWS = DATA / "marker_places_audit_rows.csv"
OUT_TOP = DATA / "marker_places_audit_top30_conflicts.csv"

In [ ]:
rows = []
for _, r in df.iterrows():
    a = _audit_places_one(r.get("docs_root"))
    rows.append(a)

audit = pd.DataFrame(rows)
out = pd.concat([df, audit], axis=1)

out["places_stable_2plus"] = (out["places_mode_count"].fillna(0) >= 2)
out["places_conflict"] = (out["places_distinct_n"].fillna(0) >= 2) & (out["places_mode_share"].fillna(1.0) < 0.75)
out["places_audit_missing"] = (out["places_mentions_total"].fillna(0) == 0)
out["places_parsed_eq_mode"] = out.apply(
    lambda x: (pd.notna(x.get("places")) and pd.notna(x.get("places_mode")) and int(float(x.get("places"))) == int(x.get("places_mode"))),
    axis=1,
)

n = len(out)
coverage = float((out["places"].notna()).mean()) if n else 0.0
stable_share = float((out["places_stable_2plus"].fillna(False)).mean()) if n else 0.0
conflict_share = float((out["places_conflict"].fillna(False)).mean()) if n else 0.0
match_share = float(out.loc[out["places_mode"].notna(), "places_parsed_eq_mode"].mean()) if int(out["places_mode"].notna().sum()) else 0.0

summary = pd.DataFrame(
    [
        {
            "rows": n,
            "places_coverage": coverage,
            "places_stable_2plus_share": stable_share,
            "places_conflict_share": conflict_share,
            "places_parsed_eq_mode_share": match_share,
        }
    ]
)

summary.to_csv(OUT_SUM, index=False)
out.to_csv(OUT_ROWS, index=False)

conf = out[out["places_conflict"]].copy()
conf["_sort"] = conf["places_distinct_n"].fillna(0) * 10 + (1 - conf["places_mode_share"].fillna(1.0))
conf = conf.sort_values(["_sort"], ascending=False).drop(columns=["_sort"]).head(30)
conf.to_csv(OUT_TOP, index=False)

print("places audit saved:")
print(" -", OUT_SUM)
print(" -", OUT_ROWS)
print(" -", OUT_TOP)
print()
print(summary.to_string(index=False))


/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'ОС 02-01'!$A:$H.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'ОС 03-01'!$A:$H.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'ОС 06-01'!$A:$H.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/диплом/.venv/lib/python3.12/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'ОС 07-01'!$A:$H.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/arinazajceva/Desktop/

places audit saved:
 - /Users/arinazajceva/Desktop/диплом/data_processed/marker_places_audit_summary.csv
 - /Users/arinazajceva/Desktop/диплом/data_processed/marker_places_audit_rows.csv
 - /Users/arinazajceva/Desktop/диплом/data_processed/marker_places_audit_top30_conflicts.csv

 rows  places_coverage  places_stable_2plus_share  places_conflict_share  places_parsed_eq_mode_share
  157         0.987261                   0.815287               0.681529                     0.642384


- `rows = 157`: в выборке 157 публикаций

- `places_coverage = 0.9873`: почти во всех строках удалось извлечь какое‑то значение `places` (вместимость/места). Примерно в 1–2 строках `places` осталось пустым.

- `places_stable_2plus_share = 0.8153`: для 82% закупок одно и то же значение мест подтвердилось минимум в двух документах (например, и в ПЗ, и в экспертизе/ТЗ). Это хороший признак, что число не случайно вытащилось из одного файла

- `places_conflict_share = 0.6815`: в 68% закупок по разным документам нашлись разные значения мест, и согласие между ними слабое (лидер не доминирует). Это не обязательно значит что-то плохое: часто в документах встречаются другие числа, которые похожи на места (дата/год, кВт, адрес “д.52”, “6 классы – 120 учащихся” и т.д.). Это просто показатель, что внутри одной закупки есть места, где можно ошибиться

- `places_parsed_eq_mode_share = 0.6424`: в 64% случаев итоговое `places`, записанное в строку, совпало с самым часто встречающимся значением по файлам этой закупки. В остальных случаях итоговое `places` либо взято из другого источника (например, из названия публикации), либо в файлах слишком много шума и “мода” получилась не тем, что надо